# 08 · Transfer attacks

Take a texture PGD found **against one network**, render it on the same
trajectory, and show it to a **different** network. Nothing changes but the
model, so how much accuracy the second network loses says how much of the
weakness the two share.

Run PGD first — `python scripts/run_attacks.py --networks all --attack pgd` —
for every network listed below. Each run's `best_texture.pt` is what gets
transferred.

Every network stays loaded at once, so this wants a GPU with room for the RAFT
models. On the command line: `python scripts/run_transfer.py --norm both`.

In [ ]:
import numpy as np, torch
from flynet_robustness import config
from flynet_robustness.models import load_networks
from flynet_robustness.attacks.transfer import save_transfer, transfer_matrix
from flynet_robustness.metrics import (bootstrap_robustness, lers, transfer_auc_matrix,
                                       transfer_target_accuracy)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
NETWORKS = config.PAPER_NETWORKS          # or a subset while you try it out
EPS = {'l2': [100, 250, 500, 1000, 2000], 'linf': [1, 6, 8, 10, 15]}
norm = 'l2'

nets = load_networks(NETWORKS, device=device)

In [ ]:
ids, acc = transfer_matrix(nets, EPS[norm], norm=norm, attacks=15, device=device)
save_transfer(f'results/Transfer/transfer_{norm}.pkl', ids, acc, EPS[norm], norm)
acc.shape      # (source, target, eps, attack); the diagonal is NaN, that is plain PGD

## How well each network holds up

`R` is the area under one network's accuracy curve, pooling the textures every
other network produced.

In [ ]:
for j, target in enumerate(ids):
    per_source = np.moveaxis(acc[:, j], 0, -1).reshape(len(EPS[norm]), -1)
    per_source = per_source[:, ~np.isnan(per_source).any(axis=0)]
    if per_source.size == 0:
        print(f'{target:16s} no textures to transfer from — run notebook 04 for the others')
        continue
    R, R_std = bootstrap_robustness(EPS[norm], per_source)
    print(f'{target:16s} transfer R = {R:.3f} ± {R_std:.3f}   LERS = {lers(R, target):.3f}')

## Which pairs share a weakness

Rows are the network the texture was made for, columns the one it was shown to.

In [ ]:
from flynet_robustness.plotting import plot_transfer_matrix
plot_transfer_matrix(transfer_auc_matrix(acc, EPS[norm]), ids);

In [ ]:
from flynet_robustness.plotting import plot_accuracy_curve
mean = transfer_target_accuracy(acc)      # (target, eps)
plot_accuracy_curve(EPS[norm], {nid: mean[j] for j, nid in enumerate(ids)},
                    xlabel=r'Perturbation size, $\epsilon$');

## Check against the published numbers

In [ ]:
import csv
ref = {r['network']: float(r['R']) for r in
       csv.DictReader(open(f'data/reference_results/Transfer_{norm}.csv'))}
for j, target in enumerate(ids):
    per_source = np.moveaxis(acc[:, j], 0, -1).reshape(len(EPS[norm]), -1)
    per_source = per_source[:, ~np.isnan(per_source).any(axis=0)]
    if per_source.size == 0 or target not in ref:
        continue
    R, _ = bootstrap_robustness(EPS[norm], per_source)
    print(f'{target:16s} ours {R:.3f}   published {ref[target]:.3f}   Δ {R - ref[target]:+.3f}')